# Auto-add ImprovMX DNS records to Cloudflare
Run All — adds 4 records to penux.uk automatically.

When prompted, paste your Cloudflare API token (it will not be stored).

In [ ]:
import requests, getpass

CF_TOKEN = getpass.getpass('Cloudflare API Token: ')
HEADERS  = {'Authorization': f'Bearer {CF_TOKEN}', 'Content-Type': 'application/json'}
BASE     = 'https://api.cloudflare.com/client/v4'

# Get Zone ID
r = requests.get(f'{BASE}/zones?name=penux.uk', headers=HEADERS, timeout=15)
d = r.json()
if not d.get('success') or not d['result']:
    print('ERROR:', d.get('errors'))
else:
    ZONE_ID = d['result'][0]['id']
    print(f'✅ Zone ID: {ZONE_ID}')

    records = [
        {'type':'MX',    'name':'penux.uk', 'content':'mx1.improvmx.com',                    'priority':10, 'ttl':1},
        {'type':'MX',    'name':'penux.uk', 'content':'mx2.improvmx.com',                    'priority':20, 'ttl':1},
        {'type':'TXT',   'name':'penux.uk', 'content':'v=spf1 include:spf.improvmx.com ~all','ttl':1},
        {'type':'CNAME', 'name':'mail',     'content':'improvmx.com', 'proxied':False,        'ttl':1},
    ]

    for rec in records:
        r = requests.post(f'{BASE}/zones/{ZONE_ID}/dns_records', headers=HEADERS, json=rec, timeout=15)
        res = r.json()
        if res.get('success'):
            x = res['result']
            print(f"✅ {x['type']:5} {x['name']} → {x['content']}")
        else:
            errs = res.get('errors', [])
            if any('already exists' in str(e) for e in errs):
                print(f"⚠️  {rec['type']} already exists — skip")
            else:
                print(f"❌ {rec['type']} {rec['name']}: {errs}")

    print('\n🎉 Done! Check: https://dash.cloudflare.com → penux.uk → DNS')
    print('\n📧 Next: register at https://improvmx.com with domain penux.uk')
    print('   Forward: netanel@penux.uk → nsh531@gmail.com')